In [ ]:
import xarray as xr
import os
import numpy as np

In [ ]:
files = [f"/work/bg1581/g260200/RUN04b_JUWELS_wpile/outputs_all/out2d_{day}.nc" for day in range(1, 366)]

In [ ]:
variables = ["windSpeedX", "windSpeedY", "elevation", "depthAverageVelX", "depthAverageVelY", "sigWaveHeight", "peakPeriod", "meanWavePeriod", "TM10",
             "meanWaveNumber", "meanWaveLength", "meanWaveDirection", "meanDirSpreading",
             "SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes", "dryFlagNode", "depth",
             "SCHISM_hgrid_node_x", "SCHISM_hgrid_node_y", "SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y",
            ]

In [ ]:
ds = xr.open_mfdataset(files, chunks="auto")

In [ ]:
ds

In [ ]:
ds = ds[variables]
for var in variables:
    ds[var] = ds[var].astype("float32")

for coord in ds.coords:
    if coord != "time":
        ds[coord] = ds[coord].astype("float32")

In [ ]:
encoding = {var: {'chunks': (24, 589428)} for var in ds.data_vars}

In [ ]:
ds

In [ ]:
zarrds = xr.open_dataset("/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_RUN04b_JUWELS_wpile.zarr", engine="zarr", chunks={})
zarrds

In [ ]:
trainds_winter = zarrds.sel(time=slice("2020-01-01", "2020-05-31"))
trainds_summer = zarrds.sel(time=slice("2020-08-01", "2020-10-31"))

In [ ]:
trainds = xr.concat([trainds_winter, trainds_summer], dim="time")
trainds.time.plot(ls="", marker="x")

In [ ]:
for var in ["windSpeedX", "windSpeedY", "elevation", "depthAverageVelX", "depthAverageVelY", "sigWaveHeight", "peakPeriod", "meanWavePeriod", "meanWaveDirection", "meanDirSpreading","depth"]:
    print(var, ":")
    print("  mean:", trainds[var].mean().values)
    print("  std:", trainds[var].std().values)

## Create CI/CD dataset

In [ ]:
variables = ["windSpeedX", "windSpeedY", "sigWaveHeight","meanWavePeriod",
            "SCHISM_hgrid_edge_nodes",  
             "SCHISM_hgrid_node_x", "SCHISM_hgrid_node_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y",
            ]

In [ ]:
cicd = zarrds.isel(time=slice(10*24, 15*24), nSCHISM_hgrid_node=slice(None, None, 100000))[variables]
cicd.to_zarr("/work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr", mode="w")

In [ ]:
!zip -r cicd.zarr.zip /work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr

In [ ]:
cicd